In [21]:
# Step 1: Install required packages
!pip -q install -U langchain-google-genai langchain-core

In [22]:
# Step 2: Enter your Gemini API key safely
import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API Key: ")

Enter your Gemini API Key: ··········


In [23]:
# Step 3: Test Gemini with LangChain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
)

response = llm.invoke([
    HumanMessage(content="Say hello in one sentence")
])

print(response.content)

Hello there!


In [24]:
#step 4 Prompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor."),
    ("human", "Explain {topic} for a beginner in {style} style.")
])

prompt_chain = prompt | llm
response = prompt_chain.invoke({
    "topic": "LangChain prompt templates",
    "style": "simple"
})
print("\nPrompt Template Output:\n", response.content)


Prompt Template Output:
 Okay, imagine you're writing a lot of similar messages or stories, but you need to change just a few details each time.

Let's say you're writing "thank you" notes. They all start with "Dear [Name]," then have a sentence about "[Gift]," and end with "Sincerely, [Your Name]."

Instead of typing out the whole thing every single time and manually changing the name and gift, wouldn't it be great to have a "fill-in-the-blanks" form?

That's *exactly* what **LangChain Prompt Templates** are for!

---

### What is a LangChain Prompt Template?

Think of it like a **"fill-in-the-blanks" form or a recipe** for the instructions you give to an AI (like ChatGPT).

It's a way to create a reusable structure for your prompts, where certain parts are left open to be filled in later.

**Key Idea:** You define the *structure* once, and then you just provide the *data* to plug into that structure.

---

### Why are they super handy?

1.  **Consistency:** Every time you use the te

In [25]:
#step 5: Simple Chain
simple_chain = prompt | llm
result = simple_chain.invoke({
    "topic": "chains in LangChain",
    "style": "technical but easy"
})
print("\nSimple Chain Output:\n", result.content)



Simple Chain Output:
 Okay, let's break down "Chains" in LangChain. Don't worry, we'll go step-by-step, using a helpful analogy!

---

### The Assembly Line Analogy

Imagine you're running a factory that makes custom products. You don't just throw raw materials at a single machine and expect a perfect finished product. Instead, you have an **assembly line**:

1.  **Station 1:** Takes raw materials, cuts them to size.
2.  **Station 2:** Assembles the basic structure.
3.  **Station 3:** Adds special features.
4.  **Station 4:** Quality checks and packaging.

Each station does a specific job, and the output of one station becomes the input for the next. This makes complex manufacturing manageable and repeatable.

---

### What are "Chains" in LangChain?

In LangChain, a **Chain** is exactly like that assembly line for your AI application.

A Large Language Model (LLM) like GPT-4 or Llama is incredibly powerful, like a super-smart general-purpose worker. But often, a single LLM call isn't

In [41]:
# Reinstall LangChain core packages to ensure all modules are available
!pip install -q -U langchain langchain-community langchain-core

In [47]:
# Step 6: Memory Example (works with current setup)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that remembers the conversation."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

memory_chain = memory_prompt | llm

print("\nMemory Example:")

user_msg_1 = "My name is Manju."
response1 = memory_chain.invoke({
    "chat_history": chat_history,
    "input": user_msg_1
})
print("AI:", response1.content)

chat_history.extend([
    HumanMessage(content=user_msg_1),
    AIMessage(content=response1.content)
])

user_msg_2 = "I am learning LangChain for my internship."
response2 = memory_chain.invoke({
    "chat_history": chat_history,
    "input": user_msg_2
})
print("AI:", response2.content)

chat_history.extend([
    HumanMessage(content=user_msg_2),
    AIMessage(content=response2.content)
])

response3 = memory_chain.invoke({
    "chat_history": chat_history,
    "input": "What is my name and what am I learning?"
})
print("AI:", response3.content)


Memory Example:
AI: Hello Manju! It's nice to meet you.
AI: That's fantastic, Manju! LangChain is a very powerful and popular framework for building applications with large language models. It's a great skill to learn for an internship.

What kind of projects are you hoping to work on with LangChain, or what aspects are you finding particularly interesting (or challenging) so far?
AI: Your name is **Manju**, and you are learning **LangChain** for your internship.


In [48]:
# 7. Tools + Agent Example
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def multiply_numbers(a: int, b: int) -> str:
    """Multiply two integers and return the result."""
    return str(a * b)

@tool
def get_topic_hint(topic: str) -> str:
    """Return a short hint about a technical topic."""
    hints = {
        "langchain": "LangChain connects LLMs with prompts, tools, memory, and data.",
        "vector store": "A vector store enables semantic similarity search.",
    }
    return hints.get(topic.lower(), "No hint available for this topic.")

agent = create_agent(
    model=llm,
    tools=[multiply_numbers, get_topic_hint],
    system_prompt="You are a helpful assistant that uses tools when needed."
)

agent_result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 12 multiplied by 9? Also give me a hint about LangChain."}
    ]
})

print("\nAgent Output:\n", agent_result)


Agent Output:
 {'messages': [HumanMessage(content='What is 12 multiplied by 9? Also give me a hint about LangChain.', additional_kwargs={}, response_metadata={}, id='1c14b72b-e992-48db-a09f-f95dba8a562b'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_topic_hint', 'arguments': '{"topic": "LangChain"}'}, '__gemini_function_call_thought_signatures__': {'94ff763c-82e4-4989-ad6d-298b2c5d1978': 'CrQCAb4+9vuGvOS15sMBa+oTLOFi+vsBaPPQin1ImCK5knG2f7kM8paJcjZt70vGvhKsXz7dc5YOdq9neuyjLA4i9L9wILVMbnj0UvFRjXQmRCuSD15wvmbhZasml6GjazUmSJd3emiLIzEYhcJ6sVIzak2JppTBccSydgc3yMN5VB5RBK4pqKfxmpqiyNtakEdPIJcc9xjIaf5Hx6DSXrDJ7dc67PD9RGQAH5J1Swo9C2VMWEPSZYlYn9duiMcJShZI568wvZWCD4LX5yidbpRM+I5j3r3903T3idAh/Zkqqo5k1gudSeu9Xsj/jmVQHnjG0VwznCCOebfzSFRriS58Fjt5uPnVqq0Ueh4ucxG+lk5r3U8dJf/8N4NelFDNIICrAwBlRL3nVE96AxaumZ85NOJ4WGw='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8143-f6

In [49]:
# Step 8: Document Loader Example

# Create a sample text file first
with open("sample_notes.txt", "w") as f:
    f.write("""LangChain is a framework for building LLM-powered applications.
It helps developers connect language models with prompts, tools, memory, and external data.
Vector stores are used to retrieve semantically similar documents.
Agents can decide which tool to use based on a user's request.
""")

from langchain_community.document_loaders import TextLoader

loader = TextLoader("sample_notes.txt")
documents = loader.load()

print("Loaded document content:\n")
print(documents[0].page_content)

Loaded document content:

LangChain is a framework for building LLM-powered applications.
It helps developers connect language models with prompts, tools, memory, and external data.
Vector stores are used to retrieve semantically similar documents.
Agents can decide which tool to use based on a user's request.



In [51]:
# Step 9: Simple vector-like retrieval demo without OpenAI API

texts = [doc.page_content for doc in documents]

query = "What helps retrieve similar documents?"

results = []
for text in texts:
    score = 0
    for word in query.lower().split():
        if word in text.lower():
            score += 1
    results.append((score, text))

results = sorted(results, reverse=True)

print("\nRetrieval Results:")
for i, (score, text) in enumerate(results[:2], 1):
    print(f"\nResult {i} (score={score}):")
    print(text)


Retrieval Results:

Result 1 (score=3):
LangChain is a framework for building LLM-powered applications.
It helps developers connect language models with prompts, tools, memory, and external data.
Vector stores are used to retrieve semantically similar documents.
Agents can decide which tool to use based on a user's request.



## Conclusion

In this assignment, I implemented and tested multiple LangChain components end‑to‑end in a Colab notebook.  
From the first cell to the last, I moved from a simple LLM call to a small but complete GenAI pipeline.

**Key takeaways:**
- Chat models like `ChatOpenAI` / Gemini can be called through a clean LangChain interface.
- `PromptTemplate` and `ChatPromptTemplate` replace hard‑coded strings and make prompts reusable and dynamic.
- Chaining prompts with the LLM lets me build modular pipelines instead of one‑off calls.
- Maintaining a conversation history acts as memory so the model can answer context‑aware questions.
- Tools and agents show how LLMs can call Python functions to perform real work beyond text generation.
- Document loaders and retrieval logic demonstrate how to bring external knowledge into the LLM workflow.

This task helped me connect the **theory of LangChain architecture** with working Python code.  
I now understand how real‑world GenAI applications are built by composing LLMs with prompts, tools, memory, and data sources in a structured way.